# Dollar Risk Pipeline — Setup Notebook

Creates and populates the `spark_risk` schema in the `prod` catalog, then registers
all tables and pipeline lineage in OpenMetadata.

**Tables**

| Table | Columns | Role |
|---|---|---|
| `prices` | trade_date, market, open, high, low, close | input |
| `positions` | trade_date, market, position | input |
| `dollar_risk` | trade_date, market, dollar_risk | output (written by Spark job) |

**Pipeline**
```
prices    ──┐
            ├──▶  DollarRiskJob (Spark)  ──▶  dollar_risk
positions ──┘
```

**Running locally** — start three port-forwards first:
```bash
kubectl port-forward -n rsch-lake svc/trino        8080:8080
kubectl port-forward -n rsch-lake svc/openmetadata 8585:8585
kubectl port-forward -n auth      svc/keycloak-keycloakx-http 8090:80
```

In [ ]:
%pip install trino pandas numpy requests networkx matplotlib --quiet

## 1. Connect to Trino and OpenMetadata

In [ ]:
import trino
import pandas as pd
import numpy as np
import requests
import base64
from datetime import date
import warnings
import networkx as nx
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
np.random.seed(42)
plt.rcParams['figure.figsize'] = (12, 4)

# ── connection settings ──────────────────────────────────────────────────────
TRINO_HOST = 'localhost'
TRINO_PORT = 8080
OM_HOST    = 'localhost'
OM_PORT    = 8585
OM_API     = f'http://{OM_HOST}:{OM_PORT}/api'
OM_EMAIL   = 'admin@open-metadata.org'
OM_PASS    = 'admin'

CATALOG   = 'prod'
SCHEMA    = 'spark_risk'
OM_SVC    = 'quarry-lake-prod'
S3_BUCKET = 'shared-lake-bucket'

# ── Keycloak JWT auth ────────────────────────────────────────────────────────
# Port-forward: kubectl port-forward -n auth svc/keycloak-keycloakx-http 8090:80
KEYCLOAK_URL   = 'http://localhost:8090/auth'
KEYCLOAK_REALM = 'rsch-lake'
CLIENT_ID      = 'PositionService'
CLIENT_SECRET  = 'gNUiRyYesntJc0o1oOG15FkDhHHMk6X9'

kc_resp = requests.post(
    f'{KEYCLOAK_URL}/realms/{KEYCLOAK_REALM}/protocol/openid-connect/token',
    data={
        'grant_type':    'client_credentials',
        'client_id':     CLIENT_ID,
        'client_secret': CLIENT_SECRET,
    },
    timeout=10,
)
kc_resp.raise_for_status()
TOKEN = kc_resp.json()['access_token']
print(f'Token obtained for {CLIENT_ID} (expires_in={kc_resp.json()["expires_in"]}s)')

# ── Trino ────────────────────────────────────────────────────────────────────
conn = trino.dbapi.connect(
    host=TRINO_HOST, port=TRINO_PORT,
    user=CLIENT_ID,
    catalog=CATALOG, schema=SCHEMA, http_scheme='http',
    auth=trino.auth.JWTAuthentication(TOKEN),
)

def q(sql, fetch=True):
    cur = conn.cursor()
    cur.execute(sql)
    if fetch:
        cols = [d[0] for d in cur.description]
        return pd.DataFrame(cur.fetchall(), columns=cols)
    cur.fetchall()
    return None

version = q('SELECT version()').iloc[0, 0]
user    = q('SELECT current_user').iloc[0, 0]
print(f'Trino {version} — connected as {user}')

# ── OpenMetadata auth ────────────────────────────────────────────────────────
resp = requests.post(
    f'{OM_API}/v1/users/login',
    json={'email': OM_EMAIL, 'password': base64.b64encode(OM_PASS.encode()).decode()},
)
resp.raise_for_status()
om_token = resp.json()['accessToken']
om = requests.Session()
om.headers.update({
    'Authorization': f'Bearer {om_token}',
    'Content-Type':  'application/json',
})
print('OpenMetadata — authenticated.')

## 2. Create Schema and Tables

In [ ]:
q(f"""
    CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}
    WITH (location = 's3://{S3_BUCKET}/{CATALOG}/{SCHEMA}/')
""", fetch=False)

tables_ddl = {
    'prices': f'''
        CREATE TABLE IF NOT EXISTS {CATALOG}.{SCHEMA}.prices (
            trade_date  DATE           COMMENT 'Trading date',
            market      VARCHAR        COMMENT 'Commodity market code',
            open        DECIMAL(12,4)  COMMENT 'Opening price',
            high        DECIMAL(12,4)  COMMENT 'Daily high price',
            low         DECIMAL(12,4)  COMMENT 'Daily low price',
            close       DECIMAL(12,4)  COMMENT 'Closing price'
        ) WITH (format = 'PARQUET', partitioning = ARRAY['market'])
    ''',
    'positions': f'''
        CREATE TABLE IF NOT EXISTS {CATALOG}.{SCHEMA}.positions (
            trade_date  DATE           COMMENT 'Trading date',
            market      VARCHAR        COMMENT 'Commodity market code',
            position    DECIMAL(12,4)  COMMENT 'Net position in lots'
        ) WITH (format = 'PARQUET', partitioning = ARRAY['market'])
    ''',
    'dollar_risk': f'''
        CREATE TABLE IF NOT EXISTS {CATALOG}.{SCHEMA}.dollar_risk (
            trade_date   DATE           COMMENT 'Trading date',
            market       VARCHAR        COMMENT 'Commodity market code',
            dollar_risk  DECIMAL(18,4)  COMMENT '21-day price vol * |position| * close'
        ) WITH (format = 'PARQUET', partitioning = ARRAY['market'])
    ''',
}

for name, ddl in tables_ddl.items():
    q(ddl, fetch=False)
    print(f'  {name} — ready')

## 3. Generate Sample Data

~252 business days (2025).  
**Prices**: geometric Brownian motion per market, OHLC derived from daily close.  
**Positions**: AR(1) process calibrated per market (φ = 0.92).

In [ ]:
MARKETS = ['CL', 'NG', 'GC', 'ZC', 'ZS']

BASE_PRICES = {
    'CL':   78.00,   # Crude Oil    (USD/barrel)
    'NG':    2.80,   # Natural Gas  (USD/MMBtu)
    'GC': 2050.00,   # Gold         (USD/oz)
    'ZC':  472.00,   # Corn         (USD/bu x100)
    'ZS': 1238.00,   # Soybeans     (USD/bu x100)
}

ANNUAL_VOL = {
    'CL': 0.35,
    'NG': 0.65,
    'GC': 0.12,
    'ZC': 0.22,
    'ZS': 0.20,
}

dates  = pd.bdate_range('2025-01-02', '2025-12-31')
n_days = len(dates)
dt     = 1 / 252

price_rows = []
for mkt in MARKETS:
    s0    = BASE_PRICES[mkt]
    sigma = ANNUAL_VOL[mkt]

    shocks  = np.random.randn(n_days)
    returns = np.exp(-0.5 * sigma**2 * dt + sigma * np.sqrt(dt) * shocks)
    returns[0] = 1.0
    close = s0 * np.cumprod(returns)

    intraday = sigma * np.sqrt(dt) * close
    open_    = close + np.random.uniform(-0.4, 0.4, n_days) * intraday
    noise_hl = np.abs(np.random.randn(n_days)) * intraday * 0.6
    high     = np.maximum(close, open_) + noise_hl
    low      = np.minimum(close, open_) - noise_hl

    for i, d in enumerate(dates):
        price_rows.append({
            'trade_date': d.date(),
            'market':     mkt,
            'open':       round(float(open_[i]),  4),
            'high':       round(float(high[i]),   4),
            'low':        round(float(low[i]),    4),
            'close':      round(float(close[i]),  4),
        })

prices_df = pd.DataFrame(price_rows)
print(f'Prices: {len(prices_df):,} rows  ({prices_df["trade_date"].min()} to {prices_df["trade_date"].max()})')
prices_df.head(3)

In [ ]:
POS_SCALE = {'CL': 40, 'NG': 80, 'GC': 15, 'ZC': 150, 'ZS': 110}

def ar1(n, phi, sigma):
    x = np.zeros(n)
    for t in range(1, n):
        x[t] = phi * x[t - 1] + sigma * np.random.randn()
    return x

pos_rows = []
for mkt in MARKETS:
    raw = ar1(n_days, phi=0.92, sigma=POS_SCALE[mkt] * 0.15)
    for i, d in enumerate(dates):
        pos_rows.append({
            'trade_date': d.date(),
            'market':     mkt,
            'position':   float(round(raw[i])),
        })

positions_df = pd.DataFrame(pos_rows)
print(f'Positions: {len(positions_df):,} rows')
positions_df.head(3)

## 4. Load Data into Trino / Iceberg

In [ ]:
for tbl in ['prices', 'positions', 'dollar_risk']:
    q(f'DELETE FROM {CATALOG}.{SCHEMA}.{tbl}', fetch=False)
    print(f'Cleared {tbl}')

def batch_insert(df, table, row_fn, batch_size=250):
    cur = conn.cursor()
    total, done = len(df), 0
    for start in range(0, total, batch_size):
        batch = df.iloc[start:start + batch_size]
        vals  = ', '.join(row_fn(r) for r in batch.itertuples())
        cur.execute(f'INSERT INTO {table} VALUES {vals}')
        cur.fetchall()
        done += len(batch)
        print(f'  {done:>5}/{total}', end='\r')
    print(f'  {done:>5}/{total} — done.')

def prices_row(r):
    return (
        f"(DATE '{r.trade_date}', '{r.market}', "
        f"DECIMAL '{r.open}', DECIMAL '{r.high}', "
        f"DECIMAL '{r.low}', DECIMAL '{r.close}')"
    )

print(f'Loading prices ({len(prices_df):,} rows)...')
batch_insert(prices_df, f'{CATALOG}.{SCHEMA}.prices', prices_row)

In [ ]:
def positions_row(r):
    return f"(DATE '{r.trade_date}', '{r.market}', DECIMAL '{int(r.position)}')"

print(f'Loading positions ({len(positions_df):,} rows)...')
batch_insert(positions_df, f'{CATALOG}.{SCHEMA}.positions', positions_row)

In [ ]:
for name in ['prices', 'positions', 'dollar_risk']:
    cnt = q(f'SELECT COUNT(*) AS c FROM {CATALOG}.{SCHEMA}.{name}').iloc[0, 0]
    print(f'  {name:15s} {cnt:>6,} rows')

## 5. Register Tables in OpenMetadata

In [ ]:
def om_put(path, payload):
    resp = om.put(f'{OM_API}/v1/{path}', json=payload)
    if not resp.ok:
        raise RuntimeError(f'PUT {path} -> {resp.status_code}: {resp.text[:400]}')
    return resp.json()

def om_get_fqn(entity_type, fqn):
    resp = om.get(f'{OM_API}/v1/{entity_type}/name/{fqn}')
    if resp.status_code == 404:
        return None
    resp.raise_for_status()
    return resp.json()

svc = om_put('services/databaseServices', {
    'name':        OM_SVC,
    'displayName': 'Trino Lakehouse',
    'serviceType': 'Trino',
    'connection': {
        'config': {
            'type':     'Trino',
            'hostPort': f'{TRINO_HOST}:{TRINO_PORT}',
            'username': 'admin',
        }
    },
})
print(f'Service  : {svc["fullyQualifiedName"]}')

db = om_put('databases', {
    'name':    CATALOG,
    'service': OM_SVC,
})
print(f'Database : {db["fullyQualifiedName"]}')

dbschema = om_put('databaseSchemas', {
    'name':     SCHEMA,
    'database': f'{OM_SVC}.{CATALOG}',
})
print(f'Schema   : {dbschema["fullyQualifiedName"]}')

In [ ]:
TABLE_COLUMNS = {
    'prices': [
        {'name': 'trade_date', 'dataType': 'DATE',    'description': 'Trading date'},
        {'name': 'market',     'dataType': 'VARCHAR',  'dataLength': 16, 'description': 'Commodity market code'},
        {'name': 'open',       'dataType': 'DECIMAL',  'description': 'Opening price'},
        {'name': 'high',       'dataType': 'DECIMAL',  'description': 'Daily high price'},
        {'name': 'low',        'dataType': 'DECIMAL',  'description': 'Daily low price'},
        {'name': 'close',      'dataType': 'DECIMAL',  'description': 'Closing price'},
    ],
    'positions': [
        {'name': 'trade_date', 'dataType': 'DATE',    'description': 'Trading date'},
        {'name': 'market',     'dataType': 'VARCHAR',  'dataLength': 16, 'description': 'Commodity market code'},
        {'name': 'position',   'dataType': 'DECIMAL',  'description': 'Net position in lots'},
    ],
    'dollar_risk': [
        {'name': 'trade_date',  'dataType': 'DATE',    'description': 'Trading date'},
        {'name': 'market',      'dataType': 'VARCHAR',  'dataLength': 16, 'description': 'Commodity market code'},
        {'name': 'dollar_risk', 'dataType': 'DECIMAL',  'description': '21-day price vol * |position| * close'},
    ],
}

schema_fqn = f'{OM_SVC}.{CATALOG}.{SCHEMA}'
table_ids  = {}

for tbl_name, columns in TABLE_COLUMNS.items():
    entity = om_put('tables', {
        'name':           tbl_name,
        'databaseSchema': schema_fqn,
        'tableType':      'Regular',
        'columns':        columns,
    })
    table_ids[tbl_name] = entity['id']
    print(f'  {tbl_name:15s} registered  (id {entity["id"][:8]}...)')

## 6. Register Spark Pipeline and Lineage

In [ ]:
spark_svc = om_put('services/pipelineServices', {
    'name':        'dollar-risk-spark',
    'displayName': 'Dollar Risk Spark Pipeline',
    'serviceType': 'CustomPipeline',
    'connection':  {'config': {'type': 'CustomPipeline'}},
})
print(f'Pipeline service : {spark_svc["fullyQualifiedName"]}')

spark_pipeline = om_put('pipelines', {
    'name':        'DollarRiskJob',
    'displayName': 'Dollar Risk Computation',
    'description': (
        'PySpark job that reads daily prices and positions from Iceberg, '
        'computes dollar risk as the 21-day rolling stddev of close price '
        'times the dollar value of the position (|position| * close).'
    ),
    'service': spark_svc['fullyQualifiedName'],
})
pipeline_id = spark_pipeline['id']
print(f'Pipeline entity  : {spark_pipeline["fullyQualifiedName"]}  (id {pipeline_id[:8]}...)')

In [ ]:
def add_edge(from_id, from_type, to_id, to_type):
    resp = om.put(f'{OM_API}/v1/lineage', json={
        'edge': {
            'fromEntity': {'id': from_id, 'type': from_type},
            'toEntity':   {'id': to_id,   'type': to_type},
        }
    })
    if not resp.ok:
        raise RuntimeError(f'{from_type} -> {to_type}: {resp.status_code}: {resp.text[:200]}')

add_edge(table_ids['prices'],    'table',    pipeline_id,              'pipeline')
print('  prices        ---->  DollarRiskJob')
add_edge(table_ids['positions'], 'table',    pipeline_id,              'pipeline')
print('  positions     ---->  DollarRiskJob')
add_edge(pipeline_id,            'pipeline', table_ids['dollar_risk'], 'table')
print('  DollarRiskJob ---->  dollar_risk')
print('\nAll lineage edges registered.')

## 7. Inspect Lineage

In [ ]:
pivot_id = table_ids['dollar_risk']
resp = om.get(
    f'{OM_API}/v1/lineage/table/{pivot_id}',
    params={'upstreamDepth': 3, 'downstreamDepth': 1},
)
resp.raise_for_status()
lineage = resp.json()

id_to_name = {n['id']: n['name'] for n in lineage.get('nodes', [])}
id_to_name[lineage['entity']['id']] = lineage['entity']['name']

all_edges = lineage.get('upstreamEdges', []) + lineage.get('downstreamEdges', [])
print('Nodes:', sorted(id_to_name.values()))
print('\nEdges:')
for e in all_edges:
    src = id_to_name.get(e['fromEntity'], e['fromEntity'])
    dst = id_to_name.get(e['toEntity'],   e['toEntity'])
    print(f'  {src:20s} ---->  {dst}')

G = nx.DiGraph()
for e in all_edges:
    src = id_to_name.get(e['fromEntity'], e['fromEntity'])
    dst = id_to_name.get(e['toEntity'],   e['toEntity'])
    G.add_edge(src, dst)

pos_layout = {
    'prices':        (-2,  1.0),
    'positions':     (-2, -1.0),
    'DollarRiskJob': ( 0,  0.0),
    'dollar_risk':   ( 2,  0.0),
}

def node_color(n):
    if n == 'DollarRiskJob': return '#e07b39'
    if n == 'dollar_risk':   return '#d7191c'
    return '#2c7bb6'

node_colors = [node_color(n) for n in G.nodes()]

fig, ax = plt.subplots(figsize=(12, 4))
nx.draw_networkx(
    G, pos_layout, ax=ax,
    node_color=node_colors, node_size=3500,
    font_size=9, font_color='white', font_weight='bold',
    edge_color='#555555', arrows=True, arrowsize=22, width=2,
)
from matplotlib.patches import Patch
ax.legend(
    handles=[
        Patch(color='#2c7bb6', label='input table'),
        Patch(color='#e07b39', label='Spark pipeline'),
        Patch(color='#d7191c', label='output table'),
    ],
    loc='lower right', fontsize=9,
)
ax.set_title('Dollar Risk Pipeline Lineage', fontsize=12, pad=12)
ax.axis('off')
plt.tight_layout()
plt.show()

## 8. Cleanup (optional)

In [ ]:
# Uncomment to drop Trino tables and schema:
# for tbl in ['dollar_risk', 'positions', 'prices']:
#     q(f'DROP TABLE IF EXISTS {CATALOG}.{SCHEMA}.{tbl}', fetch=False)
# q(f'DROP SCHEMA IF EXISTS {CATALOG}.{SCHEMA}', fetch=False)
# print('Trino tables dropped.')

# Uncomment to remove OpenMetadata entities:
# for tbl_id in table_ids.values():
#     om.delete(f'{OM_API}/v1/tables/{tbl_id}?hardDelete=true&recursive=true')
# pipeline_ent = om_get_fqn('pipelines', 'dollar-risk-spark.DollarRiskJob')
# if pipeline_ent:
#     om.delete(f'{OM_API}/v1/pipelines/{pipeline_ent["id"]}?hardDelete=true&recursive=true')
# svc_ent = om_get_fqn('services/pipelineServices', 'dollar-risk-spark')
# if svc_ent:
#     om.delete(f'{OM_API}/v1/services/pipelineServices/{svc_ent["id"]}?hardDelete=true&recursive=true')
# print('OpenMetadata entities removed.')